# Episodic Memory를 사용하는 멀티 에이전트 의료 시스템

## 소개

이 Notebook에서는 AgentCore Memory SDK와 Strands Memory Hook을 사용하여 **Episodic Memory를 갖춘 멀티 에이전트 의료 시스템**을 구현하는 방법을 보여 줍니다. 이 접근 방식을 사용하면 API를 직접 호출하지 않고도 메모리를 자동으로 관리할 수 있습니다.

### 튜토리얼 세부 정보

| 항목                 | 세부 정보                                                                         |
|:--------------------|:----------------------------------------------------------------------------------|
| 튜토리얼 유형       | Episodic Memory 및 멀티 에이전트 조율                                             |
| 에이전트 유형       | 의료 도우미 시스템                                                                |
| 에이전트 프레임워크 | Memory Hook을 사용하는 Strands Agents                                            |
| LLM 모델            | Anthropic Claude Sonnet 4                                                        |
| 튜토리얼 구성 요소  | Episodic Memory, Memory Hook, HealthLake 통합                                     |
| 예제 난이도         | 중급                                                                              |

이 튜토리얼에서 학습할 내용은 다음과 같습니다.

- Episodic Memory에 MemoryClient SDK를 활용하는 방법
- 자동 메모리 관리를 위한 Memory Hook 생성 방법
- 공유 Episodic Memory를 사용하는 전문 에이전트 구현 방법
- 실시간 HealthLake FHIR 쿼리 통합 방법

## Episodic Memory가 의료 도우미에 기여하는 방식

**EpisodicStrategy**는 상호 작용을 구조화된 에피소드로 수집하고 여러 세션에 걸쳐 의미 있는 인사이트를 생성합니다. 단순히 "무슨 일이 있었는지" 기록하는 것을 넘어 상호 작용이 "왜", "어떻게" 진행되었는지 파악합니다.

### 3단계 처리 과정

1. **추출(Extraction)**: 단기 메모리(이벤트)에서 유용한 인사이트를 식별하여 구조화된 에피소드 형태로 장기 메모리에 저장합니다.
2. **통합(Consolidation)**: 정보를 새 에피소드에 기록할지 기존 에피소드를 업데이트할지 결정합니다.
3. **리플렉션(Reflection)**: 여러 에피소드에서 인사이트를 생성하여 패턴과 개선점을 식별합니다.

### 에피소드 구조

각 에피소드는 다음 내용을 수집합니다.
- **상황(Situation)**: 의료 전문가가 달성하려던 목표
- **의도(Intent)**: 상호 작용의 주요 목표
- **평가(Assessment)**: 목표를 성공적으로 달성했는지 여부
- **근거(Justification)**: 해당 평가를 내린 이유
- **턴별 분석**: 에이전트 라우팅, 도구 사용, 의사 결정 과정을 보여 주는 상세 분석
- **에피소드 수준의 리플렉션**: 해당 세션에서 효과적이었던 요소에 관한 인사이트

### 환자 수준의 리플렉션

리플렉션은 여러 에피소드의 내용을 통합하여 더 폭넓은 인사이트를 추출합니다.
- **성공적인 전략**: 지속적으로 효과를 내는 패턴(예: 라우팅 프로토콜, 데이터 표시 방식)
- **일반적인 사용 사례**: 이 환자가 자주 문의하는 유형
- **잠재적 개선 사항**: 도우미의 효율성을 높일 수 있는 영역
- **습득한 교훈**: 여러 상호 작용에서 얻은 인사이트

### 의료 워크플로에서의 이점

1. **라우팅 개선**: 어떤 유형의 질문을 어느 에이전트가 가장 효과적으로 처리하는지 학습합니다.
2. **데이터 표시 개선**: 복잡한 의료 데이터를 빠르게 이해할 수 있도록 구성하는 방법을 파악합니다.
3. **패턴 인식**: 특정 환자의 일반적인 문의 패턴을 식별합니다.
4. **품질 개선**: 여러 세션에 걸쳐 효과가 있었던 요소와 그렇지 않은 요소를 추적합니다.
5. **컨텍스트 인식**: 과거 세션에서 얻은 교훈을 향후 상호 작용에 활용합니다.

이 튜토리얼에서는 에피소드가 멀티 에이전트 상호 작용의 전체 흐름을 수집하는 방식과 리플렉션이 의료 도우미를 지속적으로 개선할 수 있는 실행 가능한 인사이트를 제공하는 방식을 살펴봅니다.

---
## 시나리오 배경

다음 구성 요소로 이루어진 **의료 도우미 시스템**을 생성합니다.
1. 환자 질문을 라우팅하는 **Supervisor Agent**
2. 보험 및 청구를 담당하는 **Claims Agent**
3. 환자 정보를 담당하는 **Demographics Agent**
4. 처방을 담당하는 **Medication Agent**

모든 에이전트는 Memory Hook을 사용하여 대화를 Episodic Memory에 자동으로 저장합니다.

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="75%" />
</div>

## 사전 요구 사항

- Python 3.10+
- Bedrock 및 AgentCore Memory 권한이 있는 AWS 자격 증명
- Amazon HealthLake 데이터 스토어(선택 사항)

시작해 보겠습니다.

## 1단계: 환경 설정
먼저 이 Notebook을 실행하는 데 필요한 라이브러리를 가져오고 클라이언트를 정의합니다.

In [ ]:
%pip install -qr ./requirements.txt

In [ ]:
import logging
from datetime import datetime
from botocore.exceptions import ClientError
from strands import Agent, tool
from strands.hooks import HookProvider, HookRegistry
from bedrock_agentcore.memory import MemoryClient

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("healthcare-assistant")

메모리 설정에 사용할 사용자 입력값을 정의합니다.

In [ ]:
MEMORY_NAME = "healthcare_episodic_memory"
PATIENT_ID = "b2055b4d-ac17-4d94-8c5b-3395e4c334dd"
region = "us-east-1"  # 사용할 AWS 리전으로 변경합니다.
SESSION_ID = f"session_{datetime.now().strftime('%Y%m%d%H%M%S')}"
MODEL_ID = "global.anthropic.claude-sonnet-4-20250514-v1:0"  # 사용할 모델 ID로 변경합니다.

print("Memory Configuration:")
print(f"  Memory Name: {MEMORY_NAME}")
print(f"  Patient ID: {PATIENT_ID}")
print(f"  Region: {region}")
print(f"  Session ID: {SESSION_ID}")
print(f"  Model ID: {MODEL_ID}")

## 2단계: HealthLake 데이터 스토어 설정

의료 에이전트가 환자 데이터를 조회할 수 있도록 HealthLake FHIR 데이터 스토어를 설정합니다.

In [ ]:
import boto3
import requests
import time
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

# HealthLake 설정
HEALTHLAKE_REGION = input("Enter HealthLake region (or press Enter for us-east-1): ").strip() or "us-east-1"
DATASTORE_ID = input("Enter HealthLake datastore ID (or press Enter to create new): ").strip()

healthlake_client = boto3.client("healthlake", region_name=HEALTHLAKE_REGION)

# 데이터 스토어가 지정되지 않은 경우 새로 생성합니다.
if not DATASTORE_ID:
    create_new = (
        input("\nNo datastore ID provided. Create new HealthLake datastore with Synthea data? (yes/no): ")
        .strip()
        .lower()
    )

    if create_new == "yes":
        print("\nCreating HealthLake datastore...")

        # 데이터 스토어를 생성합니다.
        create_response = healthlake_client.create_fhir_datastore(
            DatastoreName=f"healthcare-demo-{int(time.time())}",
            DatastoreTypeVersion="R4",
            PreloadDataConfig={"PreloadDataType": "SYNTHEA"},
        )

        DATASTORE_ID = create_response["DatastoreId"]
        print(f"✅ Datastore created: {DATASTORE_ID}")
        print("⏳ Waiting for datastore to become ACTIVE (this may take 10-15 minutes)...")

        # ACTIVE 상태가 될 때까지 기다립니다.
        while True:
            status_response = healthlake_client.describe_fhir_datastore(DatastoreId=DATASTORE_ID)
            status = status_response["DatastoreProperties"]["DatastoreStatus"]

            if status == "ACTIVE":
                print("✅ Datastore is ACTIVE")
                break
            elif status in ["FAILED", "DELETING"]:
                print(f"❌ Datastore creation failed with status: {status}")
                raise Exception(f"Datastore creation failed: {status}")

            print(f"   Status: {status}...")
            time.sleep(30)

        print(f"\n✅ Synthea data loaded. Using default patient ID: {PATIENT_ID}")


# HealthLake 엔드포인트를 가져옵니다.
datastore = healthlake_client.describe_fhir_datastore(DatastoreId=DATASTORE_ID)
HEALTHLAKE_ENDPOINT = datastore["DatastoreProperties"]["DatastoreEndpoint"]


def query_healthlake(resource_type, search_params=None, resource_id=None):
    """HealthLake FHIR API를 쿼리합니다."""
    if resource_id:
        url = f"{HEALTHLAKE_ENDPOINT}/{resource_type}/{resource_id}"
    else:
        url = f"{HEALTHLAKE_ENDPOINT}/{resource_type}"
        if search_params:
            params = "&".join([f"{k}={v}" for k, v in search_params.items()])
            url += f"?{params}"

    session = boto3.Session()
    credentials = session.get_credentials()

    request = AWSRequest(method="GET", url=url, headers={"Accept": "application/fhir+json"})
    SigV4Auth(credentials, "healthlake", HEALTHLAKE_REGION).add_auth(request)

    response = requests.get(url, headers=dict(request.headers))

    if response.status_code == 200:
        return response.json()
    else:
        return {"error": f"Failed to fetch: {response.text}"}


print(f"\n{'=' * 70}")
print("HealthLake Configuration:")
print(f"  Datastore ID: {DATASTORE_ID}")
print(f"  Endpoint:     {HEALTHLAKE_ENDPOINT}")
print(f"  Region:       {HEALTHLAKE_REGION}")
print(f"  Patient ID:   {PATIENT_ID}")
print(f"{'=' * 70}")

## 3단계: Episodic Strategy를 사용하는 메모리 생성

각 에이전트에 하나씩 할당되는 여러 브랜치를 지원하는 단일 메모리 리소스를 생성합니다. 이 공유 메모리 리소스가 기반 역할을 하고, 각 브랜치는 에이전트 대화에 격리된 컨텍스트를 제공합니다.

하나의 리포지토리(메모리 리소스)에 여러 브랜치(에이전트 컨텍스트)가 있는 Git 리포지토리와 같은 구조입니다.

In [ ]:
client = MemoryClient(region_name=region)

strategies = [
    {
        "episodicMemoryStrategy": {
            "name": "HealthcareEpisodes",
            "description": "Captures healthcare interactions as episodes",
            "namespaceTemplates": ["/healthcare/{actorId}/{sessionId}/"],
            "reflectionConfiguration": {"namespaceTemplates": ["/healthcare/{actorId}/"]},
        }
    }
]

try:
    memory = client.create_memory_and_wait(
        name=MEMORY_NAME,
        strategies=strategies,
        description="Healthcare system with episodic memory",
        event_expiry_days=7,  # 단기 대화는 7일 후 만료됩니다.
        max_wait=300,
        poll_interval=10,
    )
    memory_id = memory["id"]
    logger.info(f"Memory created successfully with ID: {memory_id}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(e):
        # memory가 이미 존재하면 해당 ID를 가져옵니다.
        memories = client.list_memories()
        memory_id = next((m["id"] for m in memories if m["id"].startswith(MEMORY_NAME)), None)
        logger.info(f"Memory already exists. Using existing memory: {memory_id}")
except Exception as e:
    # memory 생성 중 발생한 오류를 처리합니다.
    print(f"❌ ERROR: {e}")
    import traceback

    traceback.print_exc()

    # 오류 발생 시 일부 생성된 memory를 삭제하여 정리합니다.
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

### 의료 멀티 에이전트 시스템의 메모리 브랜치 이해

생성한 메모리 리소스는 의료 멀티 에이전트 아키텍처의 핵심 기능인 **브랜칭**을 지원합니다. 작동 방식은 다음과 같습니다.

**단일 메모리 리소스와 여러 브랜치:**
- 모든 에이전트가 동일한 `memory_id`와 `session_id`를 공유합니다.
- 각 에이전트는 컨텍스트 격리를 위한 자체 `branch_name`을 갖습니다.

**의료 멀티 에이전트 시스템의 주요 이점:**

1. **컨텍스트 격리**: 각 에이전트가 서로 간섭하지 않고 자체 대화 이력을 유지합니다.
   - Claims Agent는 보험 및 청구 대화만 확인합니다.
   - Demographics Agent는 환자 정보 대화만 확인합니다.
   - Medication Agent는 처방 관련 대화만 확인합니다.
   - Supervisor Agent는 기본 라우팅 및 조율 흐름을 확인합니다.

2. **병렬 실행 안전성**: 여러 에이전트를 동시에 실행할 수 있습니다.
   - 에이전트를 병렬로 실행해도 메모리 충돌이 발생하지 않습니다.
   - 각 브랜치에 독립적으로 접근할 수 있습니다.
   - 동시 처리가 필요한 의료 워크플로에 중요한 기능입니다.

3. **명확한 감사 추적 기록**: 각 에이전트의 상호 작용을 추적할 수 있습니다.
   - 각 의료 에이전트가 논의한 내용을 확인할 수 있습니다.
   - 에이전트별 문제를 디버깅할 수 있습니다.
   - 환자 진료 대화의 흐름을 파악할 수 있습니다.
   - 규정 준수 및 문서화 요구 사항을 충족할 수 있습니다.

**의료 브랜치 구조:**
- `main` 브랜치: Supervisor Agent의 라우팅 결정
- `claims_agent` 브랜치: 보험 및 청구 대화
- `demographics_agent` 브랜치: 환자 정보 업데이트
- `medication_agent` 브랜치: 처방 관련 대화

## 4단계: 브랜치를 지원하는 Memory Hook Provider 생성

In [ ]:
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from strands.hooks import AgentInitializedEvent, MessageAddedEvent
from bedrock_agentcore.memory import MemorySessionManager


class HealthcareMemoryHooks(HookProvider):
    def __init__(self, memory_id: str, region_name: str = None, branch_name: str = "main"):
        """MemorySessionManager로 훅을 초기화합니다.

        인자:
            memory_id: AgentCore Memory ID
            region_name: 메모리 서비스용 AWS 리전
            branch_name: 이 에이전트 메모리의 브랜치 이름(기본값: "main")
        """
        if region_name is None:
            region_name = region  # 전역 region 변수를 사용합니다.

        self.memory_manager = MemorySessionManager(memory_id=memory_id, region_name=region_name)
        self.memory_id = memory_id
        self.branch_name = branch_name
        self._sessions = {}  # actor/session 조합별 세션 객체를 캐시합니다.
        self._branch_initialized = False  # 브랜치 생성 여부를 추적합니다.

    def _get_or_create_session(self, actor_id: str, session_id: str):
        """주어진 행위자와 세션의 MemorySession을 가져오거나 생성합니다."""
        key = f"{actor_id}:{session_id}"
        if key not in self._sessions:
            self._sessions[key] = self.memory_manager.create_memory_session(actor_id=actor_id, session_id=session_id)
        return self._sessions[key]

    def _initialize_branch(self, actor_id: str, session_id: str):
        """main 브랜치가 아니면서 브랜치가 없으면 초기화합니다."""
        if self._branch_initialized or self.branch_name == "main":
            return

        try:
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 브랜치가 이미 존재하는지 확인합니다.
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)

            if not branch_exists:
                # 분기 기준으로 사용할 마지막 이벤트를 main 브랜치에서 가져옵니다.
                main_events = memory_session.list_events(branch_name="main")
                if not main_events:
                    # main 브랜치에 초기 이벤트를 생성합니다.
                    memory_session.add_turns(
                        [ConversationalMessage("Healthcare system initialized", MessageRole.ASSISTANT)]
                    )
                    main_events = memory_session.list_events(branch_name="main")

                if main_events:
                    last_event = main_events[-1]
                    # 브랜치를 생성합니다.
                    memory_session.fork_conversation(
                        root_event_id=last_event.eventId,
                        branch_name=self.branch_name,
                        messages=[
                            ConversationalMessage(
                                f"Starting {self.branch_name} healthcare branch",
                                MessageRole.ASSISTANT,
                            )
                        ],
                    )
                    logger.info(f"✅ Created healthcare branch: {self.branch_name}")

            self._branch_initialized = True

        except Exception as e:
            logger.error(f"Failed to initialize healthcare branch {self.branch_name}: {e}")

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """의료 에이전트가 시작될 때 최근 대화 기록을 불러옵니다."""
        try:
            # 에이전트 상태에서 세션 정보를 가져옵니다.
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in healthcare agent state")
                return

            # main이 아닌 브랜치는 필요한 경우 초기화합니다.
            if self.branch_name != "main":
                self._initialize_branch(actor_id, session_id)

            # 메모리 세션을 가져옵니다.
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 이 브랜치에서 최근 5개 대화 턴을 가져옵니다.
            recent_turns = memory_session.get_last_k_turns(
                k=5, branch_name=self.branch_name, include_parent_branches=False
            )

            if recent_turns:
                # 에이전트의 시스템 프롬프트에 컨텍스트를 추가합니다.
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message.get("role", "unknown").lower()
                        text = message.get("content", {}).get("text", "")
                        if text:
                            context_messages.append(f"{role.title()}: {text}")

                if context_messages:
                    context = "\n".join(context_messages[-10:])  # 최근 메시지 10개
                    event.agent.system_prompt += (
                        f"\n\nRecent healthcare conversation history:\n{context}\n\n"
                        "Continue the conversation naturally based on this context."
                    )
                    logger.info(f"✅ Loaded healthcare context from branch '{self.branch_name}'")

        except Exception as e:
            logger.error(f"Failed to load healthcare conversation history: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        """의료 대화 턴을 메모리의 적절한 브랜치에 저장합니다."""
        try:
            # 에이전트 상태에서 세션 정보를 가져옵니다.
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in healthcare agent state")
                return

            # 메모리 세션을 가져옵니다.
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 마지막 메시지를 가져옵니다.
            messages = event.agent.messages
            if not messages:
                return

            last_message = messages[-1]
            role_str = last_message.get("role", "").upper()
            content_text = last_message.get("content", [{}])[0].get("text", "")

            if not content_text:
                logger.debug("Skipping empty healthcare message")
                return

            # 역할 문자열을 MessageRole enum에 매핑합니다.
            role_mapping = {
                "USER": MessageRole.USER,
                "ASSISTANT": MessageRole.ASSISTANT,
                "TOOL": MessageRole.TOOL,
            }
            message_role = role_mapping.get(role_str, MessageRole.USER)

            # 메시지를 해당 브랜치에 저장합니다.
            if self.branch_name == "main":
                # main 브랜치에는 턴을 일반적인 방식으로 추가합니다.
                memory_session.add_turns(messages=[ConversationalMessage(content_text, message_role)])
            else:
                # main이 아닌 브랜치에서는 기존 브랜치에 추가해야 합니다.
                # 브랜치가 없으면 초기화합니다.
                if not self._branch_initialized:
                    self._initialize_branch(actor_id, session_id)

                # 기존 브랜치에 추가합니다.
                memory_session.add_turns(
                    messages=[ConversationalMessage(content_text, message_role)],
                    branch={"name": self.branch_name},
                )

            logger.info(f"Memory saved to healthcare branch: {self.branch_name}")

        except Exception as e:
            logger.error(f"Failed to store healthcare message: {e}")

    def get_session(self, actor_id: str, session_id: str):
        """직접 액세스할 메모리 세션 객체를 가져옵니다."""
        return self._get_or_create_session(actor_id, session_id)

    def register_hooks(self, registry: HookRegistry) -> None:
        """의료 메모리 훅을 레지스트리에 등록합니다."""
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)


print("✅ Healthcare memory hook provider defined")

## 5단계: Memory Branching을 사용하는 멀티 에이전트 의료 아키텍처 생성

이 섹션에서는 **서로 다른 메모리 브랜치**를 사용하는 전문 의료 에이전트를 생성하여 브랜칭 기능을 보여 줍니다.

### 의료 브랜칭 전략
- **기본 브랜치(`main`)**: Supervisor Agent의 라우팅 결정을 저장하고 기본 대화 흐름 역할을 합니다.
- **`claims_agent` 브랜치**: 보험 및 청구 대화를 위한 별도의 브랜치입니다.
- **`demographics_agent` 브랜치**: 환자 정보 및 연락처 정보를 위한 별도의 브랜치입니다.
- **`medication_agent` 브랜치**: 처방 및 약물 대화를 위한 별도의 브랜치입니다.

각 전문 의료 에이전트는 자체 브랜치에서 작동하며, 처음 사용할 때 기본 대화에서 자동으로 분기됩니다. 이를 통해 다음이 가능합니다.

- 의료 전문 분야별로 독립된 대화 흐름 유지
- 도메인별 의료 컨텍스트 격리
- Supervisor Agent의 기본 대화 흐름 보존
- 의료 데이터 분리 요구 사항 준수
- 환자 상호 작용 유형별로 명확한 감사 추적 기록 유지

### 의료 에이전트 역할
- **Supervisor Agent**: 환자 질문을 적절한 전문가에게 라우팅합니다.
- **Claims Agent**: 보험 청구, 진료비 문의 및 보장 범위 질문을 처리합니다.
- **Demographics Agent**: 환자의 인구 통계 정보와 연락처 업데이트를 관리합니다.
- **Medication Agent**: 처방 질문, 복용량 정보 및 약물 관리를 처리합니다.

이 아키텍처는 일관된 환자 진료 경험을 유지하면서 민감한 의료 대화를 적절히 격리합니다.

### 브랜치 메모리를 사용하는 에이전트 생성

다음으로 시스템 프롬프트를 정의하고 서로 다른 메모리 브랜치를 사용하는 에이전트를 생성합니다. 동일한 `actor_id`와 `session_id`를 사용하면서 서로 다른 `branch_name` 값으로 격리된 대화 컨텍스트를 생성하는 방식에 주목하세요.

In [ ]:
# 의료 supervisor의 시스템 프롬프트
SUPERVISOR_PROMPT = """You are a healthcare supervisor agent. Route patient questions to:
    - Claims Agent: for insurance, billing, claims questions
    - Demographics Agent: for personal info, contact details  
    - Medication Agent: for prescriptions, medications, dosage
    
    Respond briefly and indicate which agent you're routing to."""

# 보험 청구 전문 에이전트의 시스템 프롬프트
CLAIMS_PROMPT = """You handle insurance claims. Use the get_patient_claims tool to fetch 
    current claim data from HealthLake. Answer questions about claims, billing, and coverage."""

# 인구 통계 전문 에이전트의 시스템 프롬프트
DEMOGRAPHICS_PROMPT = """You handle patient demographics. Use the get_patient_demographics tool to 
    fetch current patient data from HealthLake. Answer questions about contact details and personal information."""

# 약물 전문 에이전트의 시스템 프롬프트
MEDICATION_PROMPT = """You handle medications. Use the get_patient_medications tool to fetch 
    current medication data from HealthLake. Answer questions about prescriptions and dosages."""

## 6단계: 의료 도구 및 Memory Hook 설정

먼저 전문 의료 에이전트를 위한 HealthLake 데이터 도구와 Memory Hook을 생성합니다. 각 에이전트는 특정 브랜치 이름으로 설정된 자체 Memory Hook을 갖습니다.
- Claims Agent는 `claims_agent` 브랜치를 사용합니다.
- Demographics Agent는 `demographics_agent` 브랜치를 사용합니다.  
- Medication Agent는 `medication_agent` 브랜치를 사용합니다.
- Supervisor Agent는 `main` 브랜치를 사용합니다.

이러한 의료 에이전트를 호출하면 다음 과정이 진행됩니다.
1. Hook이 브랜치의 존재 여부를 확인합니다.
2. 브랜치가 없으면 기본 대화에서 새 브랜치로 분기합니다.
3. 에이전트의 의료 대화를 전용 브랜치에 저장합니다.
4. 각 에이전트는 환자 데이터 개인정보 보호를 위해 격리된 컨텍스트를 유지합니다.
5. HealthLake 데이터 도구를 통해 실시간 환자 정보에 접근할 수 있습니다.
6. 적절한 데이터 격리를 통해 의료 규정 준수를 유지합니다.

**HealthLake 데이터 도구:**
- `get_patient_claims`: 보험 청구 정보를 가져옵니다.
- `get_patient_demographics`: 환자 연락처와 인구 통계 데이터를 가져옵니다.
- `get_patient_medications`: 현재 처방 및 약물 데이터를 가져옵니다.

**메모리 브랜치 구조:**
- 각 에이전트는 자체 메모리 컨텍스트에서 독립적으로 작동합니다.
- 환자 대화는 의료 도메인별로 적절히 격리됩니다.
- Supervisor Agent는 분리를 유지하면서 에이전트 간 작업을 조율합니다.


In [ ]:
# 의료 Memory Hook을 None으로 초기화합니다.
supervisor_hooks = None
claims_hooks = None
demographics_hooks = None
medication_hooks = None

In [ ]:
@tool
def get_patient_claims(patient_id: str = PATIENT_ID) -> dict:
    """Get patient insurance claims from HealthLake"""
    return query_healthlake("Claim", {"patient": patient_id})


@tool
def get_patient_medications(patient_id: str = PATIENT_ID) -> dict:
    """Get patient medications from HealthLake"""
    return query_healthlake("MedicationRequest", {"patient": patient_id})


@tool
def get_patient_demographics(patient_id: str = PATIENT_ID) -> dict:
    """Get patient demographic information from HealthLake"""
    return query_healthlake("Patient", resource_id=patient_id)


# 각 에이전트의 Memory Hook을 생성합니다.
supervisor_hooks = HealthcareMemoryHooks(memory_id, region, "main")
claims_hooks = HealthcareMemoryHooks(memory_id, region, "claims_agent")
demographics_hooks = HealthcareMemoryHooks(memory_id, region, "demographics_agent")
medication_hooks = HealthcareMemoryHooks(memory_id, region, "medication_agent")

print("✅ HealthLake tools and memory hooks created")

### 의료 에이전트 생성

이제 앞에서 설정한 도구와 Memory Hook을 사용하여 의료 에이전트를 생성합니다.

- **Supervisor Agent**: 질문 라우팅(`main` 브랜치)
- **Claims Agent**: 보험 및 청구(`claims_agent` 브랜치)  
- **Demographics Agent**: 환자 정보(`demographics_agent` 브랜치)
- **Medication Agent**: 처방(`medication_agent` 브랜치)

각 에이전트에는 전문 시스템 프롬프트와 관련 도구, 대화 격리를 위한 Memory Hook이 제공됩니다.

In [ ]:
# 메모리 브랜칭을 사용하는 전문 의료 에이전트를 생성합니다.
supervisor = Agent(
    model=MODEL_ID,
    system_prompt=SUPERVISOR_PROMPT,
    hooks=[supervisor_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

claims_agent = Agent(
    model=MODEL_ID,
    system_prompt=CLAIMS_PROMPT,
    tools=[get_patient_claims],
    hooks=[claims_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

demographics_agent = Agent(
    model=MODEL_ID,
    system_prompt=DEMOGRAPHICS_PROMPT,
    tools=[get_patient_demographics],
    hooks=[demographics_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

medication_agent = Agent(
    model=MODEL_ID,
    system_prompt=MEDICATION_PROMPT,
    tools=[get_patient_medications],
    hooks=[medication_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

print("✅ Healthcare agents created with HealthLake tools and memory branching")

#### Episodic Memory를 사용하는 의료 멀티 에이전트 시스템이 준비되었습니다.

## 의료 도우미 테스트

환자 진료 시나리오로 의료 멀티 에이전트 시스템을 테스트해 보겠습니다.

**실행해 볼 예시 질문:**
- "What's the status of my insurance claims?"
- "Can you update my contact information?"
- "What medications am I currently taking?"
- "Do I have any pending billing issues?"
- "What's my current address on file?"
- "Are there any drug interactions with my prescriptions?"
- "How much do I owe for my recent visit?"
- "Can you tell me about my coverage details?"

In [ ]:
# 의료 에이전트와 대화형 채팅을 시작합니다.
print("Healthcare Assistant - Type 'quit' to exit\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["quit", "exit", "q"]:
        break

    if not user_input:
        continue

    # Supervisor Agent가 라우팅을 처리합니다.
    routing = str(supervisor(user_input))
    print(f"\nSupervisor: {routing}")

    # Supervisor Agent의 결정에 따라 적절한 에이전트로 라우팅합니다.
    if "claims agent" in routing.lower():
        response = str(claims_agent(user_input))
        print(f"\nClaims Agent: {response}\n")
    elif "demographics agent" in routing.lower():
        response = str(demographics_agent(user_input))
        print(f"\nDemographics Agent: {response}\n")
    elif "medication agent" in routing.lower():
        response = str(medication_agent(user_input))
        print(f"\nMedication Agent: {response}\n")

## 의료 메모리 브랜치 검사

AgentCore Memory Branching의 주요 이점 중 하나는 각 의료 에이전트의 대화 이력을 독립적으로 검사할 수 있다는 점입니다. 이는 다음 작업에 매우 중요합니다.

**의료 멀티 에이전트 시스템 디버깅:**
- 각 의료 에이전트가 환자와 나눈 대화 내용을 정확히 확인합니다.
- 어떤 에이전트가 어떤 의료 문의를 처리했는지 식별합니다.
- 의료 시스템에서 환자 정보가 전달되는 흐름을 추적합니다.

**의료 에이전트 조율 방식 이해:**
- 에이전트가 개별 의료 컨텍스트를 유지했는지 확인합니다.
- 동시 실행 중 환자 데이터 충돌이 발생하지 않았는지 확인합니다.
- 의료 에이전트 상호 작용의 타임라인을 검토합니다.
- 격리된 대화 추적을 통해 HIPAA 규정 준수를 보장합니다.

**의료 분야의 이점:**
- **Claims Agent**: 모든 보험 및 청구 대화를 추적합니다.
- **Demographics Agent**: 환자 정보 업데이트 및 변경 사항을 모니터링합니다.
- **Medication Agent**: 모든 처방 및 약물 대화를 점검합니다.
- **Supervisor Agent**: 라우팅 결정과 환자 분류를 검토합니다.

환자 상담 중 생성된 의료 브랜치를 살펴보겠습니다.

In [ ]:
print("\n=== Viewing Healthcare Memory Branches ===")

if claims_hooks or demographics_hooks or medication_hooks:
    # 브랜치 목록 조회에 사용할 메모리 세션을 가져옵니다. 모두 동일한 세션을 가리킵니다.
    hook = claims_hooks if claims_hooks else (demographics_hooks if demographics_hooks else medication_hooks)
    if hook:
        memory_session = hook.get_session(actor_id=PATIENT_ID, session_id=SESSION_ID)

        # 세션의 모든 브랜치를 나열합니다.
        branches = memory_session.list_branches()
        print(f"\n📊 Session has {len(branches)} branches total:")
        for branch in branches:
            events = memory_session.list_events(branch_name=branch.name)
            print(f"  - Branch: {branch.name}")
            print(f"    └─ Events: {len(events)}")
            print(f"    └─ Created: {branch.created}")

            # 이 브랜치의 최근 대화를 출력합니다.
            if events:
                print("    └─ Recent conversations:")
                for event in events[-100:]:  # 최근 이벤트 10개를 표시합니다.
                    for payload in event.payload:
                        if "conversational" in payload:
                            role = payload["conversational"]["role"]
                            text = payload["conversational"]["content"]["text"]
                            print(f"        {role}: {text[:500]}...")

        print("\n💡 Each branch represents a different agent's memory:")
        print("  • 'main' = Supervisor agent conversations")
        print("  • 'claims_agent' = Claims assistant conversations")
        print("  • 'demographics_agent' = Demographics assistant conversations")
        print("  • 'medication_agent' = Medication assistant conversations")
else:
    print("No memory hooks found. Make sure to run the cell that creates the hooks first.")

## 의료 장기 메모리 검증: 에피소드 및 리플렉션

의료 시스템이 **EpisodicStrategy**를 사용하여 단기 대화를 환자에 관한 구조화된 장기 인사이트로 변환한 방식을 살펴보겠습니다.

### 의료 에피소드
에피소드는 통합된 환자 상호 작용에서 다음 내용을 수집합니다.
- **임상 컨텍스트**: 환자 진료 목표 및 결과
- **에이전트 조율**: Supervisor Agent와 전문 에이전트가 협업한 방식
- **데이터 통합**: HealthLake 정보 검색 및 표시

### 환자 리플렉션
리플렉션은 여러 에피소드에서 다음 인사이트를 제공합니다.
- **진료 패턴**: 환자의 의사소통 선호도와 반복되는 요구 사항
- **효과적인 전략**: 이 환자에게 가장 효과적인 접근 방식
- **최적화 기회**: 향후 진료를 개선할 수 있는 영역

에피소드와 리플렉션은 비동기식으로 처리됩니다.

In [ ]:
print("=== HEALTHCARE LONG-TERM MEMORY: EPISODES & REFLECTIONS ===")
actor_id = PATIENT_ID
session_id = SESSION_ID
# 의료 에피소드와 리플렉션의 네임스페이스를 정의합니다.
episode_namespace = f"/healthcare/{actor_id}/{session_id}/"
reflection_namespace = f"/healthcare/{actor_id}/"
print(f"\n📋 Episode namespace: {episode_namespace}")
print(f"🧠 Reflection namespace: {reflection_namespace}")

try:
    print("\n📖 HEALTHCARE EPISODES (Session-specific patient interactions)")
    episodes = client.retrieve_memories(
        memory_id=memory_id,
        namespace=episode_namespace,
        query="patient healthcare interactions",
        top_k=10,
    )
    print(f"Found {len(episodes)} healthcare episode(s)")

    for i, episode in enumerate(episodes, 1):
        print(f"\n🏥 Healthcare Episode {i}:")
        content = episode.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            # 의료 컨텍스트를 확인할 수 있도록 내용을 더 표시합니다.
            print(f"   {text[:300]}..." if len(text) > 300 else f"   {text}")
        print(f"   Score: {episode.get('score', 'N/A')}")

    if not episodes:
        print("   No episodes found yet. Episodic processing happens asynchronously.")

except Exception as e:
    print(f"❌ Error retrieving healthcare episodes: {e}")

try:
    print("\n🔍 PATIENT REFLECTIONS (Cross-session healthcare insights)")
    reflections = client.retrieve_memories(
        memory_id=memory_id,
        namespace=reflection_namespace,
        query="patient care patterns and insights",
        top_k=10,
    )
    print(f"Found {len(reflections)} patient reflection(s)")

    for i, reflection in enumerate(reflections, 1):
        print(f"\n💡 Patient Reflection {i}:")
        content = reflection.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            # 의료 인사이트를 확인할 수 있도록 내용을 더 표시합니다.
            print(f"   {text[:400]}..." if len(text) > 400 else f"   {text}")
        print(f"   Score: {reflection.get('score', 'N/A')}")

    if not reflections:
        print("   No reflections found yet. Reflections are generated after multiple episodes.")

except Exception as e:
    print(f"❌ Error retrieving patient reflections: {e}")

print("\n💡 TIP: Use the memory browser for interactive healthcare memory visualization")
print("   Episodes show individual patient consultation summaries")
print("   Reflections reveal patterns in patient care and preferences")
print("\n⏱️  NOTE: Episode and reflection generation takes 10-15 minutes after conversations")
print("   Check back later if no episodes/reflections appear immediately")

## 요약

### 구현한 구성 요소
1. **Supervisor Agent**: `main` 브랜치에서 작업을 조율합니다.
2. **Claims Agent**: `claims_agent` 브랜치에서 보험 업무를 처리합니다.
3. **Demographics Agent**: `demographics_agent` 브랜치에서 환자 정보를 관리합니다.
4. **Medication Agent**: `medication_agent` 브랜치에서 약물 업무를 처리합니다.

### 메모리 아키텍처
- **단기 메모리**: 각 에이전트에 격리된 브랜치가 있습니다.
- **에피소드**: 세션별로 `/healthcare/{actorId}/{sessionId}/`에 저장됩니다.
- **리플렉션**: 모든 세션이 `/healthcare/{actorId}/`에서 공유합니다.

### 이점
- ✅ 에이전트가 서로의 대화를 방해하지 않습니다.
- ✅ 모든 에이전트가 동일한 세션의 장기 메모리에 기여합니다.
- ✅ 학습한 패턴(리플렉션)을 모든 환자 세션에서 공유합니다.
- ✅ 에이전트별 전체 대화 이력이 유지됩니다.

## 리소스 정리(선택 사항)

이 셀을 실행하여 튜토리얼에서 생성한 메모리와 IAM 역할을 삭제합니다.

In [ ]:
# import boto3

# print("Cleanup Options:")
# delete_memory = input("Delete memory? (yes/no): ").strip().lower()
# if delete_memory == 'yes':
#   try:
#       print(f"Deleting memory: {memory_id}")
#       client.delete_memory_and_wait(memory_id=memory_id)
#       print("Memory deleted")
#   except Exception as e:
#       print(f"Error deleting memory: {e}")
# else:
#   print(f"Memory preserved: {memory_id}")

# delete_healthlake = input("Delete HealthLake datastore? (yes/no): ").strip().lower()
# if delete_healthlake == 'yes':
#     try:
#         print(f"Deleting HealthLake datastore: {DATASTORE_ID}")
#         healthlake_client.delete_fhir_datastore(DatastoreId=DATASTORE_ID)
#         print("HealthLake datastore deletion initiated")
#     except Exception as e:
#         print(f"Error deleting HealthLake datastore: {e}")
# else:
#     print(f"HealthLake datastore preserved: {DATASTORE_ID}")

# print("Cleanup complete")